# DoRA Fine-Tuning on Llama-3.2-1B

Reference: Liu et al. 2024, *DoRA: Weight-Decomposed Low-Rank Adaptation* — [arXiv:2402.09353](https://arxiv.org/abs/2402.09353)


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from pathlib import Path
import sys
# ATTENTION
# Requires adding folder shortcut to MyDrive
PROJECT_ROOT = Path('/content/gdrive/MyDrive/DoRA_Final_Project')
Results = PROJECT_ROOT / 'Results'

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes torchao==0.16.0 trl datasets==2.21.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 135.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
from pathlib import Path
import os, math, time, gc, json
from dataclasses import dataclass
from typing import Tuple, Dict, Optional, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    set_seed,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


torch 2.10.0+cu128 | cuda True


## 2. Hugging Face token
The model we use(Llama-3.2-1B) is gated.
Using `HF_TOKEN` to secrets to access model

In [ ]:
def _hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = _hf_token()
assert HF_TOKEN, "Set HF_TOKEN in Colab Secrets or as an env var."
print("HF token loaded.")


HF token loaded.


## 3. Configuration
Hyper-parameters dataclass so we have identical setup across 4 variants (Baseline, LoRA, DoRA Library, DoRA Manual)

Here is table comparison of setting and reasoning behind modifications

| Setting        | Paper (Llama-7B/2-7B)  | This notebook                   |
|----------------|------------------------|---------------------------------|
| Rank           | 16 or 32               | **8**                          |
| α              | 32                     | **16**                          |
| Target modules | q, k, v, up, down      | **q, k, v, up, down**  |
| Train data     | commonsense_170k       | **commonsense_170k (20k slice)**|
| Train steps    | 1–3 epochs             | **940 steps (~3 ep of 20k)**

20,000 / 16(batch) * 4 (grad_accum) = 312.5 (1 epoch)

We use lower rank, becuase we are not seeing significance differnce between LoRA and DoRA on 1B model at rank = 16. We adjust alpha accordingly, usually alpha is recommended to be 2r or r to maintain stability. We speculate that this is due to model size and rank 16 is maybe too lenient to show difference. In the paper we saw that the margin between LoRA and DoRA shrinks after rank increase, so we did the opposite in-order to better show DoRA's performance at parameter efficiency.

Given the budget constraint and experiment setup we have to adjust training steps and evaluations coverage. We are running 4 * Eval + 3 (Training) so we shorten the training steps and coverage.

Params are set to train on A100

In [ ]:
# Claude Helped making config look nice
@dataclass
class Config:
    # ── Model ─────────────────────────────────────────────────────────────
    model_id: str = "meta-llama/Llama-3.2-1B"
    dtype:    str = "bfloat16"

    # ── LoRA / DoRA hyper-params ──────────────────────────────────────────
    rank:          int   = 8
    lora_alpha:    int   = 16    # s = alpha/rank = 2.0
    lora_dropout:  float = 0.05
    target_modules: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj",
        "up_proj", "down_proj"
    )

    # ── Training data ─────────────────────────────────────────────────────
    dataset_name: str   = "zwhe99/commonsense_170k"
    n_train:      int   = 20_000
    n_val:        int   = 500
    max_seq_len:  int   = 512
    batch_size:   int   = 16
    grad_accum:   int   = 4
    lr:           float = 2e-4
    n_steps:      int   = 940  # 3 epochs

    warmup_ratio: float = 0.1

    # Two seeds for robustness: LoRA & manual_DoRA are run at both.
    seed:         int   = 67
    alt_seed:     int   = 13

    # ── Evaluation ────────────────────────────────────────────────────────
    n_bench_samples: int = 1000
    eval_batch_size: int = 8
    eval_steps:      int = 100   # was 250 → finer-grained eval-loss curve
    logging_steps:   int = 5     # was 10  → smoother training-loss curve

    # ── I/O ───────────────────────────────────────────────────────────────
    output_dir: str = "/content/gdrive/MyDrive/DoRA_Final_Project/Results"

config = Config()
os.makedirs(config.output_dir, exist_ok=True)


## 4. The DoRA math

LoRA updates a frozen matrix $W_0$ with a low-rank residual:

$$W_{\text{LoRA}} \;=\; W_0 + s \cdot B A, \qquad s = \alpha / r$$

### Weight-Decomposed Factorization
The core idea of DoRA is to decompose the weights into a **magnitude** component and a **direction** component. The effective weight $W'$ is defined as:

$$W' = m \cdot \frac{W_0 + sBA}{\|W_0 + sBA\|_{\text{row}}}$$

* $m \in \mathbb{R}^{\text{out}}$ — trainable, one scalar per output row
* $\lVert \cdot \rVert_{\text{row}}$ — per-row L2 norm, **detached** from the backward graph (DoRA paper §4)
* $B$ starts at zero and $m$ starts at $\lVert W_0 \rVert_{\text{row}}$ — so step 0 reproduces $W_0 x$ exactly


## 5. DoRA from scratch

Drop-in replacement for a frozen `nn.Linear`. This method introduces three trainable parameters: `lora_A`, `lora_B`, and `magnitude` ($m$). The base weight $W_0$ remains frozen.

**Parameter Budget:** $\approx (r \cdot d_{in} + r \cdot d_{out}) + d_{out}$
*(If input_size = output_size: $2rd + d$ )

Defining $G = m / \|W_0 + sBA\|_{\text{row}}$ and expanding:

$$y = G(W_0 + sBA)x = W_0 x + (G-1)W_0\,\text{drop}(x) + G(sBA)\,\text{drop}(x)$$

At init $G = 1$ (since $BA=0$), so the correction terms vanish and the layer is an exact copy of $W_0 x$.

In [ ]:
class DoRALinear(nn.Module):
  def __init__(self, base_layer: nn.Linear, rank: int, lora_alpha: float, lora_dropout: float=0.0):
    super().__init__()

    self.base_layer = base_layer
    self.rank = rank
    self.scaling = lora_alpha / rank

    # Freeze base layer
    for p in self.base_layer.parameters():
      p.requires_grad = False

    in_features = base_layer.in_features
    out_features = base_layer.out_features
    dtype = base_layer.weight.dtype
    device = base_layer.weight.device

    # LoRA parameters: A (down-proj) B(up-proj)
    self.lora_A = nn.Parameter(torch.zeros((self.rank, in_features), dtype=dtype, device=device))
    self.lora_B = nn.Parameter(torch.zeros((out_features, self.rank), dtype=dtype, device=device))

    # magnitude m
    with torch.no_grad():
      W0 = base_layer.weight
      m_init = torch.linalg.norm(W0, ord=2, dim=1) # Row wise norm because of Pytorch
      self.magnitude = nn.Parameter(m_init.to(dtype).to(device))

    nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    self.lora_dropout = nn.Dropout(lora_dropout)

  def forward(self, x):
    W0 = self.base_layer.weight
    bias = self.base_layer.bias

    # No dropout, base output
    base_out = F.linear(x, W0, bias)
    drop_x = self.lora_dropout(x)

    # LoRA params
    sBA = self.scaling * (self.lora_B @ self.lora_A)

    # Clamp prevent division by 0
    with torch.no_grad():
      norm_V = torch.linalg.vector_norm(W0 + sBA.detach(), ord=2, dim=1, keepdim=True).clamp_min(1e-12)


    # G = m / ||V||
    g = self.magnitude.view(-1, 1) / norm_V

    direction = F.linear(drop_x, W0) * (g - 1.0).view(-1)

    lora_out    = F.linear(drop_x, self.lora_A)              # (batch, rank)
    lora_out    = F.linear(lora_out, self.lora_B)            # (batch, out)
    adapter_out = lora_out * self.scaling * g.view(-1)        # (batch, out)

    return base_out + direction + adapter_out

In [ ]:
def count_trainable_params(model: nn.Module) -> Tuple[int, int, float]:
    """Return (trainable, total, pct%)."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    pct       = 100.0 * trainable / total if total > 0 else 0.0
    return trainable, total, pct


def _matches_any(name: str, patterns: Tuple[str, ...]) -> bool:
    """True if `name` exactly equals or ends with any pattern."""
    return any(name == p or name.endswith(f".{p}") for p in patterns)

# Prevents memory leak when testing different PEFT methods
def free_memory(*objs) -> None:
    """Delete objects and release GPU cache between sequential runs."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Base Model
def load_base_model(config: Config):
    model = AutoModelForCausalLM.from_pretrained(
        config.model_id,
        torch_dtype=getattr(torch, config.dtype),
        device_map="auto",
        attn_implementation="sdpa"
    )

    tokenizer = AutoTokenizer.from_pretrained(config.model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer


def prepare_datasets(config):
    ds = load_dataset(config.dataset_name, split="train")
    # filter
    def is_valid(ex):
        output = ex.get("output", "")
        return isinstance(output, str) and len(output.strip()) > 0

    ds = ds.filter(is_valid)
    # format
    def format_prompt(ex):
        instruction = ex.get("instruction", "")
        inp = ex.get("input", "")
        output = ex.get("output", "")

        if inp:
            prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
        else:
            prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

        full_text = prompt + output

        return {
            "text": full_text,
            "prompt": prompt,
            "response": output,
        }

    ds = ds.map(format_prompt, remove_columns=ds.column_names)
    # shuffle
    ds = ds.shuffle(seed=config.seed)
    # Split
    split = ds.train_test_split(
        test_size=config.n_val,
        seed=config.seed
    )
    train_data = split["train"].select(range(config.n_train))
    val_data = split["test"]

    print(f"Train: {len(train_data)} | Val: {len(val_data)}")
    return train_data, val_data

In [ ]:
BENCH_DATASETS: Dict[str, Tuple[str, str, Optional[str]]] = {
    "boolq":         ("google/boolq",                   "validation", None),
    "hellaswag":     ("Rowan/hellaswag",                "validation", None),
    "winogrande":    ("winogrande",                     "validation", "winogrande_xl"),
    "arc_easy":      ("allenai/ai2_arc",                "validation", "ARC-Easy"),
    "arc_challenge": ("allenai/ai2_arc",                "validation", "ARC-Challenge"),
    "openbookqa":    ("allenai/openbookqa",             "validation", "main"),
    "piqa":          ("ybisk/piqa",                     "validation", None),
    "siqa":          ("allenai/social_i_qa",            "validation", None),
}

def _load_bench_dataset(repo_id: str, split: str = "validation", config_name: Optional[str] = None):
    if config_name is None:
        return load_dataset(repo_id, split=split, trust_remote_code=True)
    return load_dataset(repo_id, config_name, split=split, trust_remote_code=True)

def _normalize_text(text: Any) -> str:
    return " ".join(str(text).strip().split())

def _first_existing(example: dict, keys: Tuple[str, ...], default: str = "") -> str:
    for key in keys:
        if key in example and example[key] is not None:
            return str(example[key])
    return default

def _device_of(model: nn.Module) -> torch.device:
    return next(model.parameters()).device

def _build_tokenized_batch(tokenizer, prompt: str, continuation: str, max_seq_len: int):
    prompt = _normalize_text(prompt)
    continuation = _normalize_text(continuation)
    if continuation:
        continuation = " " + continuation

    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    cont_ids = tokenizer(continuation, add_special_tokens=False).input_ids

    input_ids = (prompt_ids + cont_ids)[:max_seq_len]
    prompt_len = min(len(prompt_ids), len(input_ids))
    labels = input_ids.copy()
    for i in range(prompt_len):
        if i < len(labels):
            labels[i] = -100

    if len(input_ids) == 0:
        input_ids = [tokenizer.eos_token_id]
        labels = [tokenizer.eos_token_id]

    # Guard against accidental prompt truncation swallowing the whole target.
    if all(l == -100 for l in labels):
        labels[-1] = input_ids[-1]

    attention_mask = [1] * len(input_ids)
    return input_ids, attention_mask, labels

@torch.no_grad()
def _score_candidate(model: nn.Module, tokenizer, prompt: str, continuation: str, max_seq_len: int) -> float:
    input_ids, attention_mask, labels = _build_tokenized_batch(tokenizer, prompt, continuation, max_seq_len)

    device = _device_of(model)
    batch = {
        "input_ids": torch.tensor([input_ids], device=device),
        "attention_mask": torch.tensor([attention_mask], device=device),
        "labels": torch.tensor([labels], device=device),
    }

    outputs = model(**batch)
    # outputs.loss is mean token-level cross-entropy over the unmasked continuation tokens.
    return float(-outputs.loss.item())

def _choice_argmax(scores: List[float]) -> int:
    return int(np.argmax(np.asarray(scores, dtype=np.float64)))

def _sample_lm_metrics(model: nn.Module, tokenizer, texts: List[str], max_seq_len: int, batch_size: int) -> Dict[str, float]:
    model.eval()
    device = _device_of(model)

    total_nll = 0.0
    total_tokens = 0

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        tokenized = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_len,
        )
        input_ids = tokenized["input_ids"].to(device)
        attention_mask = tokenized["attention_mask"].to(device)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        n_tokens = int((labels != -100).sum().item())
        total_nll += float(outputs.loss.item()) * n_tokens
        total_tokens += n_tokens

    avg_loss = total_nll / max(total_tokens, 1)
    ppl = float(math.exp(avg_loss)) if avg_loss < 50 else float("inf")
    return {"loss": avg_loss, "perplexity": ppl, "tokens": float(total_tokens)}

def lm_eval(model: nn.Module, tokenizer, val_data, config: Config, name: str = "model") -> Dict[str, float]:
    texts = [_normalize_text(ex["text"]) for ex in val_data]
    metrics = _sample_lm_metrics(model, tokenizer, texts, config.max_seq_len, config.eval_batch_size)
    print(f"[{name}] val loss = {metrics['loss']:.4f} | ppl = {metrics['perplexity']:.2f} | tokens = {int(metrics['tokens'])}")
    return metrics

def _boolq_prompt(ex: dict) -> Tuple[str, List[str], int]:
    passage = _first_existing(ex, ("passage", "paragraph", "text"))
    question = _first_existing(ex, ("question",))
    label = int(ex["answer"]) if "answer" in ex else int(ex["label"])
    prompt = f"Passage: {passage}\nQuestion: {question}\nAnswer yes or no:"
    return prompt, ["yes", "no"], label

def _hellaswag_prompt(ex: dict) -> Tuple[str, List[str], int]:
    ctx_a = _first_existing(ex, ("ctx_a",))
    ctx_b = _first_existing(ex, ("ctx_b",))
    ctx = _first_existing(ex, ("ctx",))
    if ctx:
        prompt_ctx = ctx
    else:
        prompt_ctx = (ctx_a + " " + ctx_b).strip()
    endings = list(ex["endings"])
    label = int(ex["label"])
    prompt = f"Context: {prompt_ctx}\nBest ending:"
    return prompt, endings, label

def _winogrande_prompt(ex: dict) -> Tuple[str, List[str], int]:
    sent = _first_existing(ex, ("sentence", "sentence1"))
    option1 = _first_existing(ex, ("option1",))
    option2 = _first_existing(ex, ("option2",))
    answer = ex.get("answer", ex.get("label"))
    answer_str = str(answer)
    label = 0 if answer_str in {"1", "A", "a", "0"} else 1
    prompt = f"Sentence: {sent}\nFill the blank with the best option:"
    return prompt, [option1, option2], label

def _arc_prompt(ex: dict) -> Tuple[str, List[str], int]:
    question = _first_existing(ex, ("question", "question_stem"))
    choices = ex["choices"]
    texts = list(choices["text"])
    answer_key = _first_existing(ex, ("answerKey",))
    labels = list(choices["label"])
    label = labels.index(answer_key) if answer_key in labels else int(answer_key)
    prompt = f"Question: {question}\nAnswer:"
    return prompt, texts, label

def _openbookqa_prompt(ex: dict) -> Tuple[str, List[str], int]:
    question = _first_existing(ex, ("question_stem", "question"))
    choices = ex["choices"]
    texts = list(choices["text"])
    answer_key = _first_existing(ex, ("answerKey",))
    labels = list(choices["label"])
    label = labels.index(answer_key) if answer_key in labels else int(answer_key)
    prompt = f"Question: {question}\nAnswer:"
    return prompt, texts, label

def _piqa_prompt(ex: dict) -> Tuple[str, List[str], int]:
    goal = _first_existing(ex, ("goal",))
    sol1 = _first_existing(ex, ("sol1",))
    sol2 = _first_existing(ex, ("sol2",))
    label = int(ex["label"])
    prompt = f"Goal: {goal}\nBest solution:"
    return prompt, [sol1, sol2], label

def _siqa_prompt(ex: dict) -> Tuple[str, List[str], int]:
    context = _first_existing(ex, ("context",))
    question = _first_existing(ex, ("question",))
    ans_a = _first_existing(ex, ("answerA",))
    ans_b = _first_existing(ex, ("answerB",))
    ans_c = _first_existing(ex, ("answerC",))
    label = int(ex["label"])
    prompt = f"Context: {context}\nQuestion: {question}\nBest answer:"
    return prompt, [ans_a, ans_b, ans_c], label

PROMPT_BUILDERS = {
    "boolq": _boolq_prompt,
    "hellaswag": _hellaswag_prompt,
    "winogrande": _winogrande_prompt,
    "arc_easy": _arc_prompt,
    "arc_challenge": _arc_prompt,
    "openbookqa": _openbookqa_prompt,
    "piqa": _piqa_prompt,
    "siqa": _siqa_prompt,
}

def evaluate_commonsense_benchmarks(model: nn.Module, tokenizer, config: Config, name: str = "model") -> Dict[str, float]:
    model.eval()
    results: Dict[str, float] = {}

    for bench_name, (repo_id, split, config_name) in BENCH_DATASETS.items():
        ds = _load_bench_dataset(repo_id, split=split, config_name=config_name)
        if config.n_bench_samples is not None and len(ds) > config.n_bench_samples:
            ds = ds.select(range(config.n_bench_samples))

        builder = PROMPT_BUILDERS[bench_name]
        correct = 0
        total = 0

        for ex in ds:
            prompt, choices, gold = builder(ex)
            scores = [_score_candidate(model, tokenizer, prompt, choice, config.max_seq_len) for choice in choices]
            pred = _choice_argmax(scores)
            correct += int(pred == gold)
            total += 1

        acc = correct / max(total, 1)
        results[bench_name] = acc
        print(f"[{name}] {bench_name:<14} acc = {acc:.4f} ({correct}/{total})")

    results["average"] = float(np.mean([results[k] for k in BENCH_DATASETS.keys()])) if BENCH_DATASETS else float("nan")
    print(f"[{name}] commonsense avg acc = {results['average']:.4f}")
    return results

# Benchmark and datasets are found mostly on hugging face

In [ ]:
def build_peft_model(base_model, config: Config, use_dora=False):
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=config.rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=list(config.target_modules),
        use_dora=use_dora,
        inference_mode=False,
        init_lora_weights=True,
    )

    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()
    return model

In [ ]:
def apply_manual_dora(model: nn.Module, config: Config, verbose: bool = True):
    targets = tuple(config.target_modules)
    n_patched = 0

    # Patch Linear layers in-place, preserving the original module object as `base_layer`.
    for _, parent in model.named_modules():
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and _matches_any(child_name, targets):
                setattr(
                    parent,
                    child_name,
                    DoRALinear(
                        base_layer=child,
                        rank=config.rank,
                        lora_alpha=config.lora_alpha,
                        lora_dropout=config.lora_dropout,
                    ),
                )
                n_patched += 1

    # Freeze everything except DoRA adapter params.
    for pname, p in model.named_parameters():
        p.requires_grad_(any(tag in pname for tag in (".lora_A", ".lora_B", ".magnitude")))

    if verbose:
        t, tot, pct = count_trainable_params(model)
        print(
            f"[apply_manual_dora] patched {n_patched} layers "
            f"(targets={targets}, r={config.rank}, α={config.lora_alpha})\n"
            f"[apply_manual_dora] trainable: {t:,} / {tot:,}  ({pct:.4f} %)"
        )
    return model


In [ ]:
def build_trainer(model, tokenizer, train_data, val_data, config: Config):
    training_args = SFTConfig(
        output_dir=config.output_dir,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.eval_batch_size,
        gradient_accumulation_steps=config.grad_accum,
        learning_rate=config.lr,
        max_steps=config.n_steps,
        lr_scheduler_type="cosine",
        warmup_ratio=config.warmup_ratio,
        logging_steps=config.logging_steps,
        eval_strategy="steps",
        eval_steps=config.eval_steps,
        save_strategy="no",
        bf16=True,
        report_to="none",
        seed=config.seed,
        data_seed=config.seed,
        dataset_text_field="text",
        max_seq_length=config.max_seq_len,
        prediction_loss_only=True, # Prevent OOM by not accumulating logits across validation set
        gradient_checkpointing=True, # Save VRAM during training
    )


    response_template = "### Response:\n"

    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer,
        mlm=False
    )

    return SFTTrainer(
        model=model,
        train_dataset=train_data,
        eval_dataset=val_data,
        tokenizer=tokenizer,
        data_collator=collator,
        args=training_args,
    )


In [ ]:
def _snapshot_base_weights(model: nn.Module) -> Dict[str, torch.Tensor]:
    snap = {}
    for name, p in model.named_parameters():
        if any(tag in name for tag in (".lora_A", ".lora_B", ".magnitude", "lora_A", "lora_B", "magnitude")):
            continue
        snap[name] = p.detach().cpu().clone()
    return snap

def _assert_same_base_weights(before: Dict[str, torch.Tensor], after: Dict[str, torch.Tensor], prefix: str = "") -> None:
    common = before.keys() & after.keys()
    for name in common:
        if not torch.equal(before[name], after[name]):
            raise AssertionError(f"{prefix} base weight mutated: {name}")
    missing = before.keys() ^ after.keys()
    if missing:
        # A wrapped PEFT model can rename some internal parameters; that's okay.
        print(f"[{prefix}] note: parameter key mismatch after wrapping for {len(missing)} entries")


# ── Magnitude / Direction analysis (DoRA paper §4, Eq. 5–6) ───────────────
#
# For an adapted Linear layer with effective merged weight W' and frozen
# base W₀ (both shape [out_features, in_features]):
#
#     Δm_i = ||W'_i,:|| − ||W₀_i,:||                (per output unit)
#     ΔD_i = 1 − cos(W'_i,:, W₀_i,:)                (per output unit)
#
# Per-module summary uses the mean across i. The DoRA paper's claim is that
# (Δm, ΔD) across modules is *negatively* correlated for DoRA (matching FT),
# while LoRA shows a *positive* correlation — i.e. LoRA cannot adjust
# direction without proportionally changing magnitude.
#
# PRECISION NOTE: We cast everything to float32 before computing norms /
# cosines. The model trains in bf16, but bf16 has only ~7 mantissa bits.
# The signal we are measuring (ΔD = 1 − cos, often ~1e-3 for subtle layers)
# is the same order as bf16 round-off, so bf16 measurement noise would
# dominate. fp32 cost is negligible because this is one-shot, post-training,
# and operates on at most ~80 matrices.

def _extract_effective_and_base(module: nn.Module) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
    """Return (W_eff, W₀) in float32 for an adapted layer, or None if not adapted.

    Handles three cases:
      • DoRALinear        (manual implementation in this notebook)
      • peft LoRA layer   (lora_A/lora_B as ModuleDict, no magnitude)
      • peft DoRA layer   (additionally has lora_magnitude_vector)
    """
    cls_name = module.__class__.__name__

    # --- Manual DoRA -----------------------------------------------------
    if cls_name == "DoRALinear":
        with torch.no_grad():
            # .float() = upcast bf16 → fp32 for the analysis (see PRECISION NOTE).
            W0 = module.base_layer.weight.detach().float()
            A  = module.lora_A.detach().float()
            B  = module.lora_B.detach().float()
            sBA = module.scaling * (B @ A)
            V = W0 + sBA
            row_norm = torch.linalg.vector_norm(V, dim=1, keepdim=True).clamp_min(1e-12)
            m = module.magnitude.detach().float()
            W_eff = (m.view(-1, 1) / row_norm) * V
        return W_eff, W0

    # --- PEFT LoRA / DoRA -----------------------------------------------
    has_peft_lora = (
        hasattr(module, "lora_A")
        and isinstance(getattr(module, "lora_A"), nn.ModuleDict)
        and hasattr(module, "base_layer")
    )
    if not has_peft_lora:
        return None

    keys = list(module.lora_A.keys())
    if not keys:
        return None
    key = keys[0]  # almost always 'default'

    with torch.no_grad():
        # .float() — same fp32 upcast as above (see PRECISION NOTE).
        W0 = module.base_layer.weight.detach().float()
        A  = module.lora_A[key].weight.detach().float()
        B  = module.lora_B[key].weight.detach().float()
        scaling = float(module.scaling[key])
        V = W0 + scaling * (B @ A)

        m_dict = getattr(module, "lora_magnitude_vector", None)
        if m_dict is not None and key in m_dict:
            mag_layer = m_dict[key]
            if hasattr(mag_layer, "weight"):
                m = mag_layer.weight.detach().float()
            else:
                m = next(mag_layer.parameters()).detach().float()
            row_norm = torch.linalg.vector_norm(V, dim=1, keepdim=True).clamp_min(1e-12)
            W_eff = (m.view(-1, 1) / row_norm) * V
        else:
            W_eff = V

    return W_eff, W0


def analyze_magnitude_direction(
    model: nn.Module,
    target_modules: Tuple[str, ...],
) -> Dict[str, Dict[str, Any]]:
    """Compute per-row Δm and ΔD for every adapted Linear matching `target_modules`."""
    results: Dict[str, Dict[str, Any]] = {}
    for name, module in model.named_modules():
        leaf = name.rsplit(".", 1)[-1] if "." in name else name
        if leaf not in target_modules:
            continue

        extracted = _extract_effective_and_base(module)
        if extracted is None:
            continue
        W_eff, W0 = extracted

        with torch.no_grad():
            norm_eff = torch.linalg.vector_norm(W_eff, dim=1)
            norm_0   = torch.linalg.vector_norm(W0,    dim=1)
            delta_m  = (norm_eff - norm_0).cpu().numpy()
            cos      = F.cosine_similarity(W_eff, W0, dim=1).clamp(-1.0, 1.0)
            delta_d  = (1.0 - cos).cpu().numpy()

        results[name] = {
            "module_type":      leaf,
            "delta_m_per_row":  delta_m,
            "delta_d_per_row":  delta_d,
            "delta_m_mean":     float(delta_m.mean()),
            "delta_d_mean":     float(delta_d.mean()),
            "delta_m_std":      float(delta_m.std()),
            "delta_d_std":      float(delta_d.std()),
            "n_rows":           int(len(delta_m)),
        }
    return results


def _md_summary_for_json(md: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    out = {}
    for name, d in md.items():
        out[name] = {
            "module_type":  d["module_type"],
            "delta_m_mean": d["delta_m_mean"],
            "delta_d_mean": d["delta_d_mean"],
            "delta_m_std":  d["delta_m_std"],
            "delta_d_std":  d["delta_d_std"],
            "n_rows":       d["n_rows"],
        }
    return out


def run_variant(
    config: Config,
    variant: str,
    tokenizer,
    train_data,
    val_data,
) -> Dict[str, Any]:
    print(f"\n{'='*60}\n  Variant: {variant.upper()}  (seed={config.seed}, rank={config.rank})\n{'='*60}")

    set_seed(config.seed)
    torch.cuda.manual_seed_all(config.seed)

    base_model, _ = load_base_model(config)
    base_before = _snapshot_base_weights(base_model)

    if variant == "lora":
        model = build_peft_model(base_model, config, use_dora=False)
    elif variant == "dora":
        model = build_peft_model(base_model, config, use_dora=True)
    elif variant == "manual_dora":
        model = apply_manual_dora(base_model, config)
    else:
        raise ValueError(f"Unknown variant: {variant!r}")

    base_after = _snapshot_base_weights(base_model)
    _assert_same_base_weights(base_before, base_after, prefix=variant)

    trainable, total, pct = count_trainable_params(model)
    print(f"[{variant}] trainable params: {trainable:,} / {total:,}  ({pct:.4f}%)")

    trainer = build_trainer(model, tokenizer, train_data, val_data, config)
    trainer.train()
    log_history = list(trainer.state.log_history)

    eval_results = trainer.evaluate()
    masked_val_loss = eval_results["eval_loss"]
    masked_val_ppl = math.exp(masked_val_loss) if masked_val_loss < 50 else float("inf")
    lm_metrics = {"loss": masked_val_loss, "perplexity": masked_val_ppl}
    print(f"[{variant}] Masked Val Loss = {masked_val_loss:.4f} | PPL = {masked_val_ppl:.2f}")

    bench = evaluate_commonsense_benchmarks(model, tokenizer, config, name=variant)
    md = analyze_magnitude_direction(model, config.target_modules)
    print(f"[{variant}] M/D analysis: {len(md)} adapted modules")

    out = {
        "lm":           lm_metrics,
        "benchmarks":   bench,
        "params": {
            "trainable":  int(trainable),
            "total":      int(total),
            "pct":        float(pct),
        },
        "log_history":  log_history,
        "md_analysis":  md,
    }

    free_memory(trainer, model, base_model)
    del trainer, model, base_model, base_before, base_after
    free_memory()
    return out


In [ ]:
# ── Run specifications ───────────────────────────────────────────────────
# Each spec describes one full training run. Results are cached by `key` so
# that re-running main() only does work for new specs.
#
# Convention: alpha = 2 * rank (matches DoRA paper Table 5), keeping the
# scaling s = alpha/rank = 2 constant when ranks differ — this isolates
# the effect of rank from the effect of effective scaling.

from dataclasses import replace as _dc_replace

@dataclass
class RunSpec:
    variant: str       # "lora" | "dora" | "manual_dora"
    seed:    int
    rank:    int
    alpha:   int

    @property
    def key(self) -> str:
        return f"{self.variant}__s{self.seed}_r{self.rank}"

    @property
    def label(self) -> str:
        return f"{self.variant} (seed={self.seed}, rank={self.rank})"


def build_run_specs(base_config: Config) -> List[RunSpec]:
    """The full set of training runs we want to perform.

    3 seeded runs × 2 methods (LoRA, manual DoRA) × 3 ranks (8, 4, 2) = 18 runs.
    Seeds are kept consistent across ranks so rank effects and seed effects are
    orthogonal in the results table.

    Alpha = 2 * rank throughout, keeping s = alpha/rank = 2 constant so that
    only rank (not effective scaling) changes across the rank sweep.
    """
    SEEDS = [67, 42, 13]
    specs = []
    for s in SEEDS:
        # r = 8  (alpha = 16)
        specs.append(RunSpec("lora",        s, 8, 16))
        specs.append(RunSpec("manual_dora", s, 8, 16))
    for s in SEEDS:
        # r = 4  (alpha = 8)
        specs.append(RunSpec("lora",        s, 4, 8))
        specs.append(RunSpec("manual_dora", s, 4, 8))
    for s in SEEDS:
        # r = 2  (alpha = 4)
        specs.append(RunSpec("lora",        s, 2, 4))
        specs.append(RunSpec("manual_dora", s, 2, 4))
    return specs


def make_run_config(base_config: Config, spec: RunSpec) -> Config:
    """Return a copy of `base_config` with `spec`'s seed/rank/alpha overrides."""
    return _dc_replace(base_config, seed=spec.seed, rank=spec.rank, lora_alpha=spec.alpha)


def parse_run_key(key: str) -> Optional[Tuple[str, int, int]]:
    """Inverse of RunSpec.key → (variant, seed, rank), or None for non-run keys."""
    if "__" not in key or key in {"base", "__md_per_row__"}:
        return None
    variant, suffix = key.split("__", 1)
    parts = suffix.split("_")
    if len(parts) != 2 or not parts[0].startswith("s") or not parts[1].startswith("r"):
        return None
    try:
        return variant, int(parts[0][1:]), int(parts[1][1:])
    except ValueError:
        return None


def main(base_config: Config) -> Dict[str, Any]:
    """
    Full pipeline:
      1. Prepare shared train / val data once (data ordering is fixed by
         base_config.seed; per-spec seeds only re-seed the optimizer / dataloader).
      2. Baseline evaluation (cached if results.json exists).
      3. For each RunSpec: train + evaluate + M/D analysis. Cache by spec.key.
      4. Print a flat summary.
    """
    set_seed(base_config.seed)
    torch.cuda.manual_seed_all(base_config.seed)
    np.random.seed(base_config.seed)

    print("Loading data …")
    train_data, val_data = prepare_datasets(base_config)
    print(f"  train: {len(train_data):,}   val: {len(val_data):,}")

    results_path = Path(base_config.output_dir) / "dora_experiment_results.json"
    md_npz_path  = Path(base_config.output_dir) / "dora_md_per_row.npz"

    all_results: Dict[str, Any] = {}
    if results_path.exists():
        try:
            with open(results_path, "r", encoding="utf-8") as f:
                all_results = json.load(f)
        except Exception as e:
            print(f"Could not load existing results: {e}")

    # Load any previously-saved per-row M/D arrays so we can extend them.
    md_per_row_all: Dict[str, Dict[str, Dict[str, np.ndarray]]] = {}
    if md_npz_path.exists():
        try:
            npz = np.load(md_npz_path)
            for full_key in npz.files:
                # Format: "{run_key}::{module_name}::{delta_m|delta_d}"
                run_key, mod_name, kind = full_key.split("::")
                md_per_row_all.setdefault(run_key, {}).setdefault(mod_name, {})[kind] = npz[full_key]
            print(f"Loaded cached M/D per-row data for {len(md_per_row_all)} runs.")
        except Exception as e:
            print(f"Could not load existing md_npz: {e}")

    tokenizer = AutoTokenizer.from_pretrained(base_config.model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ── Baseline ─────────────────────────────────────────────────────────
    if "base" in all_results:
        print("\n----- Baseline (cached) -----")
    else:
        print("\nLoading base model for baseline …")
        base_model, _ = load_base_model(base_config)
        b_trainable, b_total, b_pct = count_trainable_params(base_model)

        print("\n----- Baseline (no training) -----")
        base_lm    = lm_eval(base_model, tokenizer, val_data, base_config, name="base")
        base_bench = evaluate_commonsense_benchmarks(base_model, tokenizer, base_config, name="base")

        free_memory(base_model)
        all_results["base"] = {
            "lm":         base_lm,
            "benchmarks": base_bench,
            "params":     {"trainable": 0, "total": int(b_total), "pct": 0.0},
        }
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=2)

    # ── Trained runs ─────────────────────────────────────────────────────
    specs = build_run_specs(base_config)
    print(f"\nWill process {len(specs)} training specs:")
    for s in specs:
        status = "✓ cached" if s.key in all_results else "  to run"
        print(f"  {status}   {s.key:<28}  ({s.label})")

    for spec in specs:
        if spec.key in all_results:
            continue
        run_cfg = make_run_config(base_config, spec)
        out = run_variant(run_cfg, spec.variant, tokenizer, train_data, val_data)

        # Persist the full per-row arrays in memory and to .npz.
        md_per_row_all[spec.key] = {
            mn: {"delta_m": d["delta_m_per_row"], "delta_d": d["delta_d_per_row"]}
            for mn, d in out["md_analysis"].items()
        }

        all_results[spec.key] = {
            "variant":     spec.variant,
            "seed":        spec.seed,
            "rank":        spec.rank,
            "alpha":       spec.alpha,
            "lm":          out["lm"],
            "benchmarks":  out["benchmarks"],
            "params":      out["params"],
            "log_history": out["log_history"],
            "md_summary":  _md_summary_for_json(out["md_analysis"]),
        }
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=2)

        # Persist md_npz incrementally so a crash mid-run doesn't lose progress.
        flat: Dict[str, np.ndarray] = {}
        for run_key, mods in md_per_row_all.items():
            for mn, arrs in mods.items():
                flat[f"{run_key}::{mn}::delta_m"] = arrs["delta_m"]
                flat[f"{run_key}::{mn}::delta_d"] = arrs["delta_d"]
        if flat:
            np.savez_compressed(md_npz_path, **flat)

    # ── Flat summary print-out ───────────────────────────────────────────
    tasks = [
        "boolq", "hellaswag", "winogrande",
        "arc_easy", "arc_challenge", "openbookqa",
        "piqa", "siqa", "average",
    ]
    cw = 8
    sorted_keys = ["base"] + sorted(
        [k for k, _ in [(k, parse_run_key(k)) for k in all_results if parse_run_key(k)] if _ is not None],
        key=lambda k: (parse_run_key(k)[0], parse_run_key(k)[2], parse_run_key(k)[1]),
    )
    header = f"{'task':<16}" + "".join(f"{k[:cw-1]:>{cw}}" for k in sorted_keys)
    print("\n" + "=" * len(header))
    print("  COMMONSENSE BENCHMARK ACCURACY  (run keys: variant__s<seed>_r<rank>)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for t in tasks:
        row = f"{t:<16}"
        for k in sorted_keys:
            val = all_results.get(k, {}).get("benchmarks", {}).get(t, float("nan"))
            row += f"{val:>{cw}.3f}"
        print(row)
    print("=" * len(header))

    print("\nLM val loss / PPL:")
    for k in sorted_keys:
        if k in all_results and "lm" in all_results[k]:
            lm = all_results[k]["lm"]
            print(f"  {k:<28}  loss={lm['loss']:.4f}  ppl={lm['perplexity']:.2f}")

    all_results["__md_per_row__"] = md_per_row_all
    return all_results

# ── Run ───────────────────────────────────────────────────────────────────
results = main(config)

Loading data …


Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/170420 [00:00<?, ? examples/s]

Filter:   0%|          | 0/170420 [00:00<?, ? examples/s]

Map:   0%|          | 0/170420 [00:00<?, ? examples/s]

...

[manual_dora] note: parameter key mismatch after wrapping for 160 entries
[manual_dora] trainable params: 1,163,264 / 1,236,977,664  (0.0940%)


/tmp/ipykernel_2460/2858932217.py:36: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  return SFTTrainer(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss,Validation Loss
100,0.162113,0.161060
200,0.154011,0.151734
300,0.141380,0.132464
400,0.127223,0.124324
500,0.103770,0.114067
600,0.099175,0.104879
700,0.084451,0.106145
800,0.081345,0.104232
900,0.082144,0.104122
940,0.094276,0.104475


Training Loss,Validation Loss,Step
0.094276,0.104475,940


[manual_dora] Masked Val Loss = 0.1045 | PPL = 1.11
[manual_dora] boolq          acc = 0.3840 (384/1000)
[manual_dora] hellaswag      acc = 0.4630 (463/1000)
[manual_dora] winogrande     acc = 0.4990 (499/1000)
[manual_dora] arc_easy       acc = 0.6316 (360/570)
[manual_dora] arc_challenge  acc = 0.3579 (107/299)
[manual_dora] openbookqa     acc = 0.3420 (171/500)
[manual_dora] piqa           acc = 0.7480 (748/1000)
[manual_dora] siqa           acc = 0.1630 (163/1000)
[manual_dora] commonsense avg acc = 0.4486
[manual_dora] M/D analysis: 80 adapted modules

  COMMONSENSE BENCHMARK ACCURACY  (run keys: variant__s<seed>_r<rank>)
task                base dora__s lora__s lora__s lora__s lora__s lora__s lora__s lora__s lora__s lora__s manual_ manual_ manual_ manual_ manual_ manual_ manual_ manual_ manual_
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
boolq      

In [ ]:
from google.colab import runtime
runtime.unassign()
